# 17. 슬라이딩 윈도우 연속 모니터링

held-out 정상과 이상 WAV를 이어 붙인 40초 신호를 10초 창·1초 hop으로 연속 판정합니다. 정상-only 대조군으로 이음새 아티팩트를 확인하고, N=1/N=3 경고의 오경고와 탐지 지연을 비교합니다.

In [ ]:
## [0] 실행 환경과 공통 상수 준비
import os, sys
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
sys.path.insert(0, '..')

import glob
import time
import librosa
import matplotlib.pyplot as plt
import koreanize_matplotlib
import numpy as np
import torch

from src import config
from src.evaluate import predict_proba
from src.model import IDClassifier
from src.preprocess import sliding_windows

MACHINE = 'id_00'
MACHINE_IDX = 0
SPLIT_SEED = 42
MODEL_SEEDS = [0, 1, 2, 3, 4]
WIN_SECONDS = 10
HOP_SECONDS = 1
WIN_LEN = config.SR * WIN_SECONDS
HOP_LEN = config.SR * HOP_SECONDS
ANOMALY_START = 20.0
ANOMALY_END = 30.0
CLIP_SEAMS = [10.0, 20.0, 30.0]
DEVICE = 'cpu'
print('추론 장치:', DEVICE)

In [ ]:
## [1] 결과를 보기 전에 seed 42 held-out 정상 파일을 정렬 순서로 고정
normalPattern = os.path.join(
    config.NOISE_DIRS['-6dB'], MACHINE, 'normal', '*.wav')
abnormalPattern = os.path.join(
    config.NOISE_DIRS['-6dB'], MACHINE, 'abnormal', '*.wav')
normalPathLST = glob.glob(normalPattern)
normalPathLST.sort()
abnormalPathLST = glob.glob(abnormalPattern)
abnormalPathLST.sort()

np.random.seed(SPLIT_SEED)
indexNP = np.random.permutation(len(normalPathLST))
TRAIN_NUM = int(len(normalPathLST) * config.TRAIN_RATIO)
testIndexNP = indexNP[TRAIN_NUM:]
heldoutNormalPathLST = []
for test_idx in testIndexNP:
    heldoutNormalPathLST.append(normalPathLST[test_idx])
heldoutNormalPathLST.sort()

SELECTED_NORMAL_PATHS = heldoutNormalPathLST[:4]
SELECTED_ABNORMAL_PATH = abnormalPathLST[0]
print('선택한 held-out 정상 파일:')
for path in SELECTED_NORMAL_PATHS:
    print(' -', os.path.basename(path))
print('선택한 이상 파일:', os.path.basename(SELECTED_ABNORMAL_PATH))

In [ ]:
## [2] 주 시나리오와 정상-only 대조군 40초 신호 생성
normalSignalLST = []
for path in SELECTED_NORMAL_PATHS:
    yNP, sr = librosa.load(path, sr=config.SR)
    if len(yNP) != WIN_LEN:
        raise ValueError(f'정상 WAV 길이가 10초가 아닙니다: {path}')
    normalSignalLST.append(yNP)

abnormalSignalNP, sr = librosa.load(
    SELECTED_ABNORMAL_PATH, sr=config.SR)
if len(abnormalSignalNP) != WIN_LEN:
    raise ValueError('이상 WAV 길이가 10초가 아닙니다.')

mainSignalNP = np.concatenate([
    normalSignalLST[0], normalSignalLST[1],
    abnormalSignalNP, normalSignalLST[2]])
controlSignalNP = np.concatenate([
    normalSignalLST[0], normalSignalLST[1],
    normalSignalLST[2], normalSignalLST[3]])
print('주 시나리오 길이(초):', len(mainSignalNP) / config.SR)
print('정상 대조군 길이(초):', len(controlSignalNP) / config.SR)

In [ ]:
## [3] 배포용 분류 앙상블·정규화값·기계별 임계값 로드
modelLST = []
for seed in MODEL_SEEDS:
    model = IDClassifier()
    modelPath = os.path.join(config.MODEL_DIR, f'idclf_{seed}.pth')
    model.load_state_dict(torch.load(
        modelPath, map_location=DEVICE, weights_only=True))
    model.eval()
    modelLST.append(model)

MIN, MAX = np.load(os.path.join(config.MODEL_DIR, 'clf_minmax.npy'))
thresholdNP = np.load(os.path.join(
    config.MODEL_DIR, 'clf_thresholds.npy'))
THRESHOLD = thresholdNP[MACHINE_IDX]

# 첫 추론 준비시간을 측정에서 제외하기 위해 모델을 한 번씩 예열한다.
warmupNP = np.zeros((1, 20032), dtype=np.float32)
for model in modelLST:
    predict_proba(model, warmupNP, DEVICE)
print('id_00 임계값:', THRESHOLD)

In [ ]:
## [4] 두 시나리오의 모든 창을 CPU에서 순차 판정하고 처리시간 측정
scenarioNameLST = ['주 시나리오', '정상 대조군']
scenarioSignalLST = [mainSignalNP, controlSignalNP]
scenarioResultDCT = {}

for scenario_idx in range(len(scenarioNameLST)):
    scenario_name = scenarioNameLST[scenario_idx]
    signalNP = scenarioSignalLST[scenario_idx]
    windowLST = sliding_windows(signalNP, WIN_LEN, HOP_LEN)
    scoreLST = []
    endTimeLST = []
    elapsedLST = []

    for window_idx in range(len(windowLST)):
        start_time = time.perf_counter()
        windowNP = windowLST[window_idx]
        melNP = librosa.feature.melspectrogram(
            y=windowNP, sr=config.SR, n_mels=config.N_MELS,
            n_fft=config.N_FFT, hop_length=config.HOP_LENGTH)
        mel_dbNP = librosa.power_to_db(melNP)
        xNP = ((mel_dbNP - MIN) / (MAX - MIN)).reshape(1, -1)

        scoreSUM = 0.0
        for model in modelLST:
            probaNP = predict_proba(model, xNP, DEVICE)
            scoreSUM = scoreSUM + (1 - probaNP[0, MACHINE_IDX])
        score = scoreSUM / len(modelLST)
        elapsed = time.perf_counter() - start_time

        scoreLST.append(score)
        elapsedLST.append(elapsed)
        endTimeLST.append(
            (window_idx * HOP_LEN + WIN_LEN) / config.SR)

    scoreNP = np.array(scoreLST)
    endTimeNP = np.array(endTimeLST)
    elapsedNP = np.array(elapsedLST)
    scenarioResultDCT[scenario_name] = {
        'scoreNP': scoreNP,
        'endTimeNP': endTimeNP,
        'elapsedNP': elapsedNP,
    }
    print(
        f'{scenario_name}: 창 {len(windowLST)}개 / '
        f'평균 {elapsedNP.mean():.4f}초 / '
        f'95퍼센타일 {np.percentile(elapsedNP, 95):.4f}초 / '
        f'최대 {elapsedNP.max():.4f}초')

In [ ]:
## [5] N=1/N=3 지속 규칙의 오경고 창과 탐지 지연 비교
mainEndTimeNP = scenarioResultDCT['주 시나리오']['endTimeNP']
mainScoreNP = scenarioResultDCT['주 시나리오']['scoreNP']
controlEndTimeNP = scenarioResultDCT['정상 대조군']['endTimeNP']
controlScoreNP = scenarioResultDCT['정상 대조군']['scoreNP']

# 각 10초 창 안에 실제 이상 구간이 얼마나 포함됐는지 비율을 계산한다.
abnormalFractionLST = []
for window_idx in range(len(mainEndTimeNP)):
    window_start = window_idx * HOP_SECONDS
    window_end = window_start + WIN_SECONDS
    overlap_start = max(window_start, ANOMALY_START)
    overlap_end = min(window_end, ANOMALY_END)
    overlap_seconds = max(0.0, overlap_end - overlap_start)
    abnormalFractionLST.append(overlap_seconds / WIN_SECONDS)
abnormalFractionNP = np.array(abnormalFractionLST)

mainNormalMaskNP = abnormalFractionNP == 0
mainAbnormalMaskNP = abnormalFractionNP > 0
print(
    '주 시나리오 이상 미포함 창 평균/최대:',
    mainScoreNP[mainNormalMaskNP].mean(),
    mainScoreNP[mainNormalMaskNP].max())
print(
    '주 시나리오 이상 포함 창 평균/최대:',
    mainScoreNP[mainAbnormalMaskNP].mean(),
    mainScoreNP[mainAbnormalMaskNP].max())
print(
    '정상 대조군 전체 창 평균/최대:',
    controlScoreNP.mean(), controlScoreNP.max())
print('정상 대조군 원본 10초 클립별 점수:')
for clip_end in [10.0, 20.0, 30.0, 40.0]:
    clip_idx = np.where(controlEndTimeNP == clip_end)[0][0]
    print(f' - 종료 {clip_end:.0f}초: {controlScoreNP[clip_idx]:.6f}')

warningResultDCT = {}
print('시나리오 | 규칙 | 오경고 창 | 오경고 알림 | 최초 유효 알림 | 탐지 지연')
print('-' * 88)
for scenario_name in scenarioNameLST:
    scoreNP = scenarioResultDCT[scenario_name]['scoreNP']
    endTimeNP = scenarioResultDCT[scenario_name]['endTimeNP']

    for required_count in [1, 3]:
        consecutive_count = 0
        warningFlagNP = np.zeros(len(scoreNP), dtype=bool)
        for window_idx in range(len(scoreNP)):
            if scoreNP[window_idx] > THRESHOLD:
                consecutive_count = consecutive_count + 1
            else:
                consecutive_count = 0
            if consecutive_count >= required_count:
                warningFlagNP[window_idx] = True

        # 연속 초과 구간에서는 첫 창에서만 알림 1건이 발생한 것으로 센다.
        alertTriggerNP = np.zeros(len(scoreNP), dtype=bool)
        for window_idx in range(len(warningFlagNP)):
            previous_warning = False
            if window_idx > 0:
                previous_warning = warningFlagNP[window_idx - 1]
            if warningFlagNP[window_idx] and not previous_warning:
                alertTriggerNP[window_idx] = True

        FIRST_VALID_ALERT = np.nan
        DETECTION_DELAY = np.nan
        if scenario_name == '주 시나리오':
            falseWindowCount = int(np.sum(
                warningFlagNP & (abnormalFractionNP == 0)))
            falseAlertCount = int(np.sum(
                alertTriggerNP & (abnormalFractionNP == 0)))
            for window_idx in range(len(alertTriggerNP)):
                if alertTriggerNP[window_idx] and abnormalFractionNP[window_idx] > 0:
                    FIRST_VALID_ALERT = endTimeNP[window_idx]
                    DETECTION_DELAY = FIRST_VALID_ALERT - ANOMALY_START
                    break
        else:
            falseWindowCount = int(np.sum(warningFlagNP))
            falseAlertCount = int(np.sum(alertTriggerNP))

        result_key = (scenario_name, required_count)
        warningResultDCT[result_key] = {
            'warningFlagNP': warningFlagNP,
            'alertTriggerNP': alertTriggerNP,
            'falseWindowCount': falseWindowCount,
            'falseAlertCount': falseAlertCount,
            'firstValidAlert': FIRST_VALID_ALERT,
            'detectionDelay': DETECTION_DELAY,
        }
        print(
            f'{scenario_name} | N={required_count} | '
            f'{falseWindowCount} | {falseAlertCount} | '
            f'{FIRST_VALID_ALERT} | '
            f'{DETECTION_DELAY}')

In [ ]:
## [6] 주 시나리오 시간축 점수·임계값·N=3 경고 시각화
n3FlagNP = warningResultDCT[('주 시나리오', 3)]['warningFlagNP']
n3TriggerNP = warningResultDCT[('주 시나리오', 3)]['alertTriggerNP']
FIRST_N3_WARNING = warningResultDCT[
    ('주 시나리오', 3)]['firstValidAlert']

plt.figure(figsize=(12, 6))
plt.plot(
    mainEndTimeNP, mainScoreNP, marker='o', linewidth=2,
    label='앙상블 이상점수')
plt.axhline(
    THRESHOLD, color='tab:red', linestyle='--',
    label=f'기계별 임계값 {THRESHOLD:.4f}')
plt.axvspan(
    ANOMALY_START, ANOMALY_END, color='tab:red', alpha=0.12,
    label='실제 이상 클립 구간')

for seam_time in CLIP_SEAMS:
    plt.axvline(
        seam_time, color='gray', linestyle=':', alpha=0.7)

exceedMaskNP = mainScoreNP > THRESHOLD
plt.scatter(
    mainEndTimeNP[exceedMaskNP], mainScoreNP[exceedMaskNP],
    color='tab:orange', s=45, zorder=4, label='임계값 초과 창')
plt.plot(
    mainEndTimeNP, abnormalFractionNP, color='gray',
    linestyle='-.', alpha=0.7, label='창 내 이상 비율')

falseAlertMaskNP = n3TriggerNP & (abnormalFractionNP == 0)
plt.scatter(
    mainEndTimeNP[falseAlertMaskNP], mainScoreNP[falseAlertMaskNP],
    marker='X', s=150, color='black', zorder=5,
    label='N=3 정상 구간 오경고 알림')

if not np.isnan(FIRST_N3_WARNING):
    warning_idx = np.where(mainEndTimeNP == FIRST_N3_WARNING)[0][0]
    plt.scatter(
        [FIRST_N3_WARNING], [mainScoreNP[warning_idx]], marker='*',
        s=260, color='crimson', edgecolor='black', zorder=5,
        label=f'N=3 최초 유효 알림 {FIRST_N3_WARNING:.0f}초')

plt.xlim(WIN_SECONDS - 0.5, 40.5)
plt.ylim(-0.03, 1.05)
plt.xlabel('창 종료시간(초) — 실제 판정 가능 시점')
plt.ylabel('점수 / 창 내 이상 비율')
plt.title('튜닝 전 슬라이딩 윈도우 모니터링 — 오경고 포함')
plt.grid(alpha=0.25)
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig(
    '../assets/sliding_window_demo.png', dpi=120,
    bbox_inches='tight')
plt.show()

## 결과 요약

- 고정 held-out 정상: `00000001.wav`, `00000004.wav`, `00000013.wav`, `00000014.wav`; 이상: `00000000.wav`.
- 40초 신호에서 10초 창·1초 hop으로 **31개 창**을 생성했습니다.
- 주 시나리오 최초 유효 알림: N=1은 22초(2초 지연), N=3은 24초(4초 지연).
- 오경고: 주 시나리오 N=1/N=3 모두 1개 알림 에피소드. 정상 대조군은 N=1 4개, N=3 3개로 지속 규칙만으로 제거되지 않았습니다.
- 주 시나리오 점수 평균은 이상 미포함 창 0.243, 이상 포함 창 0.781이며 이상 구간에서 상승했습니다. 정상 대조군 평균은 0.154였습니다.
- 정상 원본 10초 클립 점수도 0.455/0.009/0.020/0.178로 일부가 임계값 0.0526을 넘었습니다. 오경고는 이음새만이 아니라 held-out 정상 분포 차이도 원인입니다.
- CPU 전처리+5모델 추론은 모든 창에서 1초 hop 이내였습니다.
- 판정: 이상 구간 점수 상승은 확인했지만, 현재 학습 정상 90퍼센타일 임계값은 held-out 연속·혼합 창에 과민합니다. 정상 연속 데이터 기반 임계값 재검증 없이는 실시간 경보 규칙을 확정할 수 없습니다.